### Change the file names

In [1]:
# Add row information to the event list file
import pandas as pd

## PARDS_Risk_V2_PatientWeights
file_path = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V2/PARDS_Risk_RNN/PARDS_Risk_V2_PatientWeights/Study23_Tag225_EventList.csv"

# Read the CSV file
df = pd.read_csv(file_path)

# Create a new column for row numbers
df.insert(0, 'Row Number', [f"Row_{i+1}" for i in range(len(df))])

# Save the modified DataFrame back to the same path
df.to_csv(file_path, index=False)

# Display the modified DataFrame
print(df.head())


  Row Number   Mark  Patient ID  Tag ID study_name  study_id tag_name  \
0      Row_1  39628        1288     225  PARDS_LWT        23     Test   
1      Row_2  39629        1748     225  PARDS_LWT        23     Test   
2      Row_3  39630        1748     225  PARDS_LWT        23     Test   
3      Row_4  39631        2294     225  PARDS_LWT        23     Test   
4      Row_5  39632        7028     225  PARDS_LWT        23     Test   

           Time Start UTC           Time Stop UTC           Time Start  \
0  2023-12-01 16:47:45+00  2023-12-04 16:47:45+00  2023-12-01 11:47:45   
1  2024-02-21 07:44:01+00  2024-02-24 07:44:01+00  2024-02-21 02:44:01   
2  2024-11-05 05:42:01+00  2024-11-08 05:42:01+00  2024-11-05 00:42:01   
3  2024-05-20 21:22:37+00  2024-05-23 21:22:37+00  2024-05-20 17:22:37   
4  2024-12-12 17:56:18+00  2024-12-15 17:56:18+00  2024-12-12 12:56:18   

             Time Stop                                    Patient Hx Link  
0  2023-12-04 11:47:45  /apps/46b9d48d-e

In [2]:
import pandas as pd
import os
import glob
import shutil
import re

# Define the file paths
directory_path = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V2/PARDS_Risk_RNN/PARDS_Risk_V2_PatientWeights/Study23_Tag225_EventList"

renamed_directory = os.path.join(directory_path, "Renamed")

# Ensure the renamed directory exists
os.makedirs(renamed_directory, exist_ok=True)

event_list_path = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V2/PARDS_Risk_RNN/PARDS_Risk_V2_PatientWeights/Study23_Tag225_EventList.csv"

# Read the event list CSV
event_df = pd.read_csv(event_list_path, dtype={"Row Number": str})

# Create a mapping from Row Number to PatientID and StartTime
event_mapping = {str(row["Row Number"]): (row["Patient ID"], row["Time Start"]) for _, row in event_df.iterrows()}

# Get all CSV files in the directory
csv_files = glob.glob(os.path.join(directory_path, "*.csv"))

for file_path in csv_files:
    file_name = os.path.basename(file_path)
    
    # Extract the row number from the filename using regex
    match = re.search(r"Row_\d+", file_name)
    if match:
        row_key = match.group()
        
        # Check if extracted row number is in the mapping
        if row_key in event_mapping:
            patient_id, start_time = event_mapping[row_key]
            
            # Convert StartTime to required format
            try:
                formatted_time = pd.to_datetime(start_time).strftime("%Y%m%d_%H")
                cat_name = "V2_PatientWeights"
                new_file_name = f"{patient_id}_{formatted_time}_{cat_name}.csv"
                new_file_path = os.path.join(renamed_directory, new_file_name)
                
                # Copy the file instead of moving it
                shutil.copy2(file_path, new_file_path)
                print(f"Copied and renamed: {file_name} -> {new_file_path}")
            except Exception as e:
                print(f"Error processing file {file_name}: {e}")


Copied and renamed: Event_Row_229_Data_zero_order_interpolation.csv -> /nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V2/PARDS_Risk_RNN/PARDS_Risk_V2_PatientWeights/Study23_Tag225_EventList/Renamed/1362933_20240826_11_V2_PatientWeights.csv
Copied and renamed: Event_Row_169_Data_zero_order_interpolation.csv -> /nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V2/PARDS_Risk_RNN/PARDS_Risk_V2_PatientWeights/Study23_Tag225_EventList/Renamed/1117361_20240306_03_V2_PatientWeights.csv
Copied and renamed: Event_Row_250_Data_zero_order_interpolation.csv -> /nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V2/PARDS_Risk_RNN/PARDS_Risk_V2_PatientWeights/Study23_Tag225_EventList/Renamed/1454681_20241105_03_V2_PatientWeights.csv
Copied and renamed: Event_Row_258_Data_zero_order_interpolation.csv -> /nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V2/PARDS_Risk_RNN/PARDS_Risk_V2_PatientWeights/Study23_Tag225_EventList/Renamed/1484434_20250128_17_V2_Patient

### Calculate the average patient weights and add into PARDS_Risk_V2_df

In [2]:
import os
import pandas as pd
from openpyxl import load_workbook

# === Paths ===
csv_folder = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V2/PARDS_Risk_RNN/PARDS_Risk_V2_PatientWeights/Study23_Tag225_EventList/Renamed"
excel_path = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V2/PARDS_Risk_RNN/PARDS_Risk_V2_df.xlsx"

# Load and clean Sheet12
df_excel = pd.read_excel(excel_path, sheet_name="Sheet12").copy()
df_excel["PID"] = df_excel["PID"].astype(str).str.strip()
df_excel["Avg_Weight"] = pd.NA

# Loop through each CSV
for filename in os.listdir(csv_folder):
    if filename.endswith(".csv"):
        pid = filename.split("_")[0].strip()
        file_path = os.path.join(csv_folder, filename)

        try:
            df = pd.read_csv(file_path)
            if "WEIGHT" in df.columns and not df["WEIGHT"].dropna().empty:
                avg_weight = df["WEIGHT"].mean()
                df_excel.loc[df_excel["PID"] == pid, "Avg_Weight"] = avg_weight
                print(f"✅ Updated PID {pid} with avg weight {avg_weight:.2f}")
            else:
                print(f"⚠️ No valid WEIGHT for {filename}")
        except Exception as e:
            print(f"❌ Error reading {filename}: {e}")

# Save as Sheet13
book = load_workbook(excel_path)
with pd.ExcelWriter(excel_path, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
    writer.book = book
    df_excel.to_excel(writer, sheet_name="Sheet13", index=False)

print("✅ All done. Sheet13 saved with Avg_Weight.")


✅ Updated PID 15673 with avg weight 11.11
⚠️ No valid WEIGHT for 1357131_20241010_17_V2_PatientWeights.csv
✅ Updated PID 509329 with avg weight 9.15
✅ Updated PID 1469711 with avg weight 78.08
✅ Updated PID 1031528 with avg weight 79.42
✅ Updated PID 606288 with avg weight 28.73
⚠️ No valid WEIGHT for 522290_20240701_13_V2_PatientWeights.csv
✅ Updated PID 1018441 with avg weight 13.83
✅ Updated PID 1370656 with avg weight 55.96
✅ Updated PID 658798 with avg weight 58.15
⚠️ No valid WEIGHT for 1106283_20240502_13_V2_PatientWeights.csv
✅ Updated PID 116930 with avg weight 19.17
⚠️ No valid WEIGHT for 1011066_20240701_07_V2_PatientWeights.csv
✅ Updated PID 1492572 with avg weight 4.69
✅ Updated PID 1016336 with avg weight 5.53
✅ Updated PID 84839 with avg weight 29.16
✅ Updated PID 21072 with avg weight 10.83
✅ Updated PID 1037492 with avg weight 5.76
⚠️ No valid WEIGHT for 1301993_20240830_20_V2_PatientWeights.csv
✅ Updated PID 1166142 with avg weight 43.30
⚠️ No valid WEIGHT for 1211379

### Calculate the Tidal Volume: TV (mL/kg) = eVT (mL) / Weight (kg)

In [3]:
# For all 276 files of V2
import os
import pandas as pd
from openpyxl import load_workbook

# === INPUT PATHS ===
csv_dir = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V2/PARDS_Risk_RNN/V2_1st_Raw/OSI_V2_1st/TWs_V2_1st/PEEP_Settings_V2_1st/Abnormal_Deletion_V2_1st"
excel_path = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V2/PARDS_Risk_RNN/PARDS_Risk_V2_df.xlsx"
sheet_name = "Sheet18"

# === OUTPUT PATH ===
output_csv_dir = os.path.join(csv_dir, "TV_V2_1st")
os.makedirs(output_csv_dir, exist_ok=True)

# === Load Excel and prepare Sheet14 ===
df_excel = pd.read_excel(excel_path, sheet_name=sheet_name)
df_excel["FileName_V2_1st"] = df_excel["FileName_V2_1st"].astype(str).str.strip()
df_sheet14 = df_excel.copy()

# Add TV columns
df_sheet14["TV_mean(mL/Kg)"] = pd.NA
df_sheet14["TV_std(mL/Kg)"] = pd.NA
for i in range(1, 7):
    df_sheet14[f"TV_mean_TW{i}(mL/Kg)"] = pd.NA
    df_sheet14[f"TV_std_TW{i}(mL/Kg)"] = pd.NA

# eVT column candidates
evt_columns = [
    "CAPSULE_AVEAA_VITAL_2325",
    "CAPSULE_DRAGERMEDIBUS_VITAL_2325",
    "CAPSULE_MAQUETC_VITAL_2325"
]

# === Process each CSV ===
for filename in os.listdir(csv_dir):
    if not filename.endswith(".csv"):
        continue

    basename = filename.replace(".csv", "").strip()
    filepath = os.path.join(csv_dir, filename)
    df = pd.read_csv(filepath)

    match = df_sheet14[df_sheet14["FileName_V2_1st"] == basename]
    if match.empty or "Tumbling_window" not in df.columns:
        continue

    # Find eVT column
    evt_col = next((col for col in evt_columns if col in df.columns and not df[col].dropna().empty), None)
    if not evt_col:
        continue

    # Get usable weight
    weight = match["Avg_Weight_Kg"].values[0]
    if pd.isna(weight) or weight == 0:
        weight = match["Weight_Kg"].values[0]
    if pd.isna(weight) or weight == 0:
        continue

    # Compute TV for each row
    df["TV (mL/Kg)"] = df[evt_col] / weight

    # Whole-file mean/std
    tv_all = df["TV (mL/Kg)"].dropna()
    if not tv_all.empty:
        tv_mean = round(tv_all.mean(), 2)
        tv_std = round(tv_all.std(), 2)
        df_sheet14.loc[match.index, "TV_mean(mL/Kg)"] = tv_mean
        df_sheet14.loc[match.index, "TV_std(mL/Kg)"] = tv_std

    # Per-tumbling-window mean/std
    for tw in range(1, 7):
        df_tw = df[df["Tumbling_window"] == tw]
        if df_tw.empty or df_tw["TV (mL/Kg)"].dropna().empty:
            continue
        tv_mean_tw = round(df_tw["TV (mL/Kg)"].mean(), 2)
        tv_std_tw = round(df_tw["TV (mL/Kg)"].std(), 2)
        df_sheet14.loc[match.index, f"TV_mean_TW{tw}(mL/Kg)"] = tv_mean_tw
        df_sheet14.loc[match.index, f"TV_std_TW{tw}(mL/Kg)"] = tv_std_tw

    # Save updated CSV
    df.to_csv(os.path.join(output_csv_dir, filename), index=False)

# === Load Excel file and append/replace sheet ===
with pd.ExcelWriter(excel_path, engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
    df_sheet14.to_excel(writer, sheet_name="Sheet19", index=False)

print("✅ Done! Sheet19 successfully written.")



In [3]:
# For all 242 files of V1
import os
import pandas as pd
from openpyxl import load_workbook

# === INPUT PATHS ===
csv_dir = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V1/VitalData/01_Vital_Raw_282_1st/OSI_285_1st/TWs_284_1st/PEEP_Settings_284_1st/Abnormal_Detection_282_1st"
excel_path = "/nfs/turbo/med-kayvan-lab/Projects/PARDS/02-Data/PARDS_Risk_V2/PARDS_Risk_RNN/PARDS_Risk_V1_df.xlsx"

# === SHEET CONFIGURATION ===
sheet_name_read = "Sheet9"
sheet_name_write = "Sheet10"

# === OUTPUT PATH ===
output_csv_dir = os.path.join(csv_dir, "TV_V1_1st")
os.makedirs(output_csv_dir, exist_ok=True)

# === Load Excel and copy to Sheet10 ===
df_excel = pd.read_excel(excel_path, sheet_name=sheet_name_read)
df_excel["FileName_Vital_1st"] = df_excel["FileName_Vital_1st"].astype(str).str.strip()
df_sheet10 = df_excel.copy()

# Add TV columns
df_sheet10["TV_mean(mL/Kg)"] = pd.NA
df_sheet10["TV_std(mL/Kg)"] = pd.NA
for i in range(1, 7):
    df_sheet10[f"TV_mean_TW{i}(mL/Kg)"] = pd.NA
    df_sheet10[f"TV_std_TW{i}(mL/Kg)"] = pd.NA

# eVT columns for V1 data
evt_columns = ["CDGR - eVT", "AVEA - eVT", "SVU - eVT"]

# === Process CSVs ===
for filename in os.listdir(csv_dir):
    if not filename.endswith(".csv"):
        continue

    basename = filename
    filepath = os.path.join(csv_dir, filename)
    df = pd.read_csv(filepath)

    match = df_sheet10[df_sheet10["FileName_Vital_1st"] == basename]
    if match.empty or "Tumbling_window" not in df.columns:
        continue

    evt_col = next((col for col in evt_columns if col in df.columns and not df[col].dropna().empty), None)
    if not evt_col:
        continue

    # Use only Avg_Weight_Kg
    weight = match["Avg_Weight_Kg"].values[0]
    if pd.isna(weight) or weight == 0:
        continue

    # Compute TV column
    df["TV (mL/Kg)"] = df[evt_col] / weight

    # Whole file mean/std
    tv_all = df["TV (mL/Kg)"].dropna()
    if not tv_all.empty:
        tv_mean = round(tv_all.mean(), 2)
        tv_std = round(tv_all.std(), 2)
        df_sheet10.loc[match.index, "TV_mean(mL/Kg)"] = tv_mean
        df_sheet10.loc[match.index, "TV_std(mL/Kg)"] = tv_std

    # Tumbling-window mean/std
    for tw in range(1, 7):
        df_tw = df[df["Tumbling_window"] == tw]
        if df_tw.empty or df_tw["TV (mL/Kg)"].dropna().empty:
            continue
        tv_mean_tw = round(df_tw["TV (mL/Kg)"].mean(), 2)
        tv_std_tw = round(df_tw["TV (mL/Kg)"].std(), 2)
        df_sheet10.loc[match.index, f"TV_mean_TW{tw}(mL/Kg)"] = tv_mean_tw
        df_sheet10.loc[match.index, f"TV_std_TW{tw}(mL/Kg)"] = tv_std_tw

    # Save CSV with added TV columns
    df.to_csv(os.path.join(output_csv_dir, filename), index=False)

# === Save updated Excel ===
with pd.ExcelWriter(excel_path, engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
    df_sheet10.to_excel(writer, sheet_name=sheet_name_write, index=False)


print("✅ Completed: TV values written to CSVs and Excel Sheet10.")


✅ Completed: TV values written to CSVs and Excel Sheet10.
